In [8]:
!pip install pyspark -q

In [9]:
import os
import zipfile
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, lpad, floor

outer_zip_path = "/content/BTS_Data_2025.zip"
outer_extract_dir = "/content/bts_data"
os.makedirs(outer_extract_dir, exist_ok=True)

# Extract outer zip
with zipfile.ZipFile(outer_zip_path, 'r') as zip_ref:
    zip_ref.extractall(outer_extract_dir)

print("Outer zip extracted.")
print("Contents of outer_extract_dir:")
for item in os.listdir(outer_extract_dir):
    print(item)

Outer zip extracted.
Contents of outer_extract_dir:
BTS_Data_2025
__MACOSX
extracted_monthly


In [10]:
for root, dirs, files in os.walk(outer_extract_dir):
    print("Folder:", root)
    print("Files:", files)
    print("-----")

Folder: /content/bts_data
Files: []
-----
Folder: /content/bts_data/BTS_Data_2025
Files: ['On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_12.zip', 'On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_6.zip', 'On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_9.zip', 'On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_2.zip', 'On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_10.zip', 'On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_4.zip', 'On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_3.zip', 'On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_5.zip', 'On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_7.zip', 'On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_1.zip', 'On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_11.zip', 'On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_8.zip', 'On_Time_Reporting_Carrier_On_T

In [11]:
# Folder that contains the monthly zip files
monthly_zip_dir = "/content/bts_data/BTS_Data_2025"

# Folder to extract all monthly zips
monthly_extract_dir = "/content/bts_data/extracted_monthly"
os.makedirs(monthly_extract_dir, exist_ok=True)

# Collect all inner zip files, and ignore duplicate files like "... 2.zip"
monthly_zip_files = [
    f for f in os.listdir(monthly_zip_dir)
    if f.endswith(".zip") and " 2" not in f
]

print("Monthly zip files to process:", len(monthly_zip_files))
for f in monthly_zip_files:
    print(f)

# Extract each monthly zip into its own subfolder
for zip_file in monthly_zip_files:
    zip_path = os.path.join(monthly_zip_dir, zip_file)

    # Use zip filename (without extension) as subfolder name
    folder_name = os.path.splitext(zip_file)[0].replace(" ", "_")
    target_dir = os.path.join(monthly_extract_dir, folder_name)
    os.makedirs(target_dir, exist_ok=True)

    try:
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(target_dir)
        print(f"Extracted: {zip_file}")
    except Exception as e:
        print(f"Failed to extract {zip_file}: {e}")

Monthly zip files to process: 12
On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_12.zip
On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_6.zip
On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_9.zip
On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_2.zip
On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_10.zip
On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_4.zip
On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_3.zip
On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_5.zip
On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_7.zip
On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_1.zip
On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_11.zip
On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_8.zip
Extracted: On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_12.zip
Extracted: On_Time_Reporting_Carrier_On_Ti

In [12]:
csv_files = []

for root, dirs, files in os.walk(monthly_extract_dir):
    for file in files:
        full_path = os.path.join(root, file)
        if file.endswith(".csv") and "__MACOSX" not in full_path and "/._" not in full_path:
            csv_files.append(full_path)

csv_files = sorted(csv_files)

print("Valid CSV files found:", len(csv_files))
print("First few CSV files:")
for f in csv_files[:5]:
    print(f)

Valid CSV files found: 12
First few CSV files:
/content/bts_data/extracted_monthly/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_1/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_1/On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2024_1.csv
/content/bts_data/extracted_monthly/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_10/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_10/On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2024_10.csv
/content/bts_data/extracted_monthly/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_11/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_11/On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2024_11.csv
/content/bts_data/extracted_monthly/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_12/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_12/On_Time_Reporting_Carrier_On_Time_Performance_(1987_prese

In [13]:
spark = SparkSession.builder \
    .appName("BTS_Airline_Delay_Cleaning") \
    .getOrCreate()

print("Spark session is ready.")

Spark session is ready.


In [14]:
df = spark.read.csv(
    csv_files,
    header=True,
    inferSchema=True
)

# Preview data
df.show(5, truncate=False)

# Print schema
df.printSchema()

# Check dataset size
print("Row count:", df.count())
print("Column count:", len(df.columns))
print("All columns:")
for c in df.columns:
    print(c)

+----+-------+-----+----------+---------+----------+-----------------+------------------------+---------------------------+-----------+-------------------------------+---------------+------------------+------------------+------+--------------+-----------+---------------+---------------+---------+-------------+----------------+----------------+----+------------+---------+-------------+-------------+-------+----------+-------+--------+---------------+--------+--------------------+----------+-------+---------+--------+------+----------+-------+--------+---------------+--------+------------------+----------+---------+----------------+--------+--------------+-----------------+-------+-------+--------+-------------+------------+------------+--------+-------------+-----------------+------------+-------------+---------------+------------------+--------------+--------------------+-----------+-----------+-----------+-------------+----------------+------------+--------------+----------------+----

In [15]:
core_columns = [
    "Year", "Quarter", "Month", "DayofMonth", "DayOfWeek",
    "FlightDate",

    "Reporting_Airline", "DOT_ID_Reporting_Airline", "IATA_CODE_Reporting_Airline",

    "Origin", "OriginCityName", "OriginState",
    "Dest", "DestCityName", "DestState",

    "CRSDepTime", "DepTime", "DepDelay", "DepDelayMinutes",
    "CRSArrTime", "ArrTime", "ArrDelay", "ArrDelayMinutes",

    "Cancelled", "CancellationCode", "Diverted",

    "CRSElapsedTime", "ActualElapsedTime", "AirTime", "Distance",

    "CarrierDelay", "WeatherDelay", "NASDelay",
    "SecurityDelay", "LateAircraftDelay"
]

df_core = df.select(*core_columns)

print("Selected columns:", len(df_core.columns))
df_core.show(5, truncate=False)

Selected columns: 35
+----+-------+-----+----------+---------+----------+-----------------+------------------------+---------------------------+------+--------------+-----------+----+------------+---------+----------+-------+--------+---------------+----------+-------+--------+---------------+---------+----------------+--------+--------------+-----------------+-------+--------+------------+------------+--------+-------------+-----------------+
|Year|Quarter|Month|DayofMonth|DayOfWeek|FlightDate|Reporting_Airline|DOT_ID_Reporting_Airline|IATA_CODE_Reporting_Airline|Origin|OriginCityName|OriginState|Dest|DestCityName|DestState|CRSDepTime|DepTime|DepDelay|DepDelayMinutes|CRSArrTime|ArrTime|ArrDelay|ArrDelayMinutes|Cancelled|CancellationCode|Diverted|CRSElapsedTime|ActualElapsedTime|AirTime|Distance|CarrierDelay|WeatherDelay|NASDelay|SecurityDelay|LateAircraftDelay|
+----+-------+-----+----------+---------+----------+-----------------+------------------------+---------------------------+--

In [16]:
from pyspark.sql.functions import col, when, lpad, floor

df_clean = df_core \
    .withColumn("is_cancelled", col("Cancelled").cast("int")) \
    .withColumn("is_diverted", col("Diverted").cast("int")) \
    .withColumn("is_dep_delayed", when(col("DepDelayMinutes") > 15, 1).otherwise(0)) \
    .withColumn("is_arr_delayed", when(col("ArrDelayMinutes") > 15, 1).otherwise(0)) \
    .withColumn("CRSDepTime_PADDED", lpad(col("CRSDepTime").cast("string"), 4, "0")) \
    .withColumn("dep_hour", floor(col("CRSDepTime_PADDED").substr(1, 2).cast("int"))) \
    .withColumn(
        "season",
        when(col("Month").isin(12, 1, 2), "Winter")
        .when(col("Month").isin(3, 4, 5), "Spring")
        .when(col("Month").isin(6, 7, 8), "Summer")
        .otherwise("Fall")
    )

df_clean = df_clean.dropna(subset=[
    "FlightDate", "Reporting_Airline", "Origin", "Dest",
    "CRSDepTime", "CRSArrTime", "Distance"
])

df_clean = df_clean.filter(col("is_cancelled") == 0)
df_clean = df_clean.filter(col("Distance") > 0)
df_clean = df_clean.filter((col("dep_hour") >= 0) & (col("dep_hour") <= 23))

df_clean = df_clean.withColumn(
    "DepDelayMinutes",
    when(col("DepDelayMinutes").isNull(), 0).otherwise(col("DepDelayMinutes"))
)

df_clean = df_clean.withColumn(
    "ArrDelayMinutes",
    when(col("ArrDelayMinutes").isNull(), 0).otherwise(col("ArrDelayMinutes"))
)

delay_cols = ["CarrierDelay", "WeatherDelay", "NASDelay", "SecurityDelay", "LateAircraftDelay"]
for c in delay_cols:
    df_clean = df_clean.withColumn(
        c,
        when(col(c).isNull(), 0).otherwise(col(c))
    )

df_clean = df_clean.withColumn(
    "is_weekend",
    when(col("DayOfWeek").isin(6, 7), 1).otherwise(0)
)

df_clean = df_clean.withColumn(
    "dep_period",
    when((col("dep_hour") >= 5) & (col("dep_hour") < 12), "Morning")
    .when((col("dep_hour") >= 12) & (col("dep_hour") < 17), "Afternoon")
    .when((col("dep_hour") >= 17) & (col("dep_hour") < 21), "Evening")
    .otherwise("Night")
)

df_clean = df_clean.withColumn(
    "is_long_flight",
    when(col("Distance") >= 1500, 1).otherwise(0)
)

In [17]:
df_clean.createOrReplaceTempView("airline_data")
print("SQL table 'airline_data' created.")

SQL table 'airline_data' created.


In [18]:
print("Cleaned row count:", df_clean.count())
print("Cleaned column count:", len(df_clean.columns))

Cleaned row count: 6969618
Cleaned column count: 45


In [19]:
df_clean.select(
    "Year", "Month", "DayOfWeek",
    "Reporting_Airline",
    "Origin", "Dest",
    "CRSDepTime", "dep_hour",
    "DepDelayMinutes", "ArrDelayMinutes",
    "is_dep_delayed", "is_arr_delayed",
    "is_cancelled", "is_diverted",
    "season"
).show(10, False)

+----+-----+---------+-----------------+------+----+----------+--------+---------------+---------------+--------------+--------------+------------+-----------+------+
|Year|Month|DayOfWeek|Reporting_Airline|Origin|Dest|CRSDepTime|dep_hour|DepDelayMinutes|ArrDelayMinutes|is_dep_delayed|is_arr_delayed|is_cancelled|is_diverted|season|
+----+-----+---------+-----------------+------+----+----------+--------+---------------+---------------+--------------+--------------+------------+-----------+------+
|2024|1    |1        |9E               |LGA   |OMA |856       |8       |0.0            |0.0            |0             |0             |0           |0          |Winter|
|2024|1    |2        |9E               |LGA   |OMA |856       |8       |0.0            |0.0            |0             |0             |0           |0          |Winter|
|2024|1    |3        |9E               |LGA   |OMA |856       |8       |0.0            |0.0            |0             |0             |0           |0          |Winter

In [20]:
for c in ["Origin", "Dest", "Distance", "DepDelayMinutes", "ArrDelayMinutes"]:
    print(c, df_clean.filter(col(c).isNull()).count())

Origin 0
Dest 0
Distance 0
DepDelayMinutes 0
ArrDelayMinutes 0


### Overall Flight Delay Pattern

This section provides a general overview of flight delay behavior in the dataset.  
It measures the average departure delay, average arrival delay, and the proportion of delayed flights.  
This helps establish a baseline understanding of overall airline delay patterns in the United States.

In [21]:
spark.sql("""
SELECT
    AVG(DepDelayMinutes) AS avg_dep_delay,
    AVG(ArrDelayMinutes) AS avg_arr_delay,
    SUM(is_dep_delayed) / COUNT(*) AS dep_delay_rate,
    SUM(is_arr_delayed) / COUNT(*) AS arr_delay_rate
FROM airline_data
""").show()

+-----------------+-----------------+-------------------+-------------------+
|    avg_dep_delay|    avg_arr_delay|     dep_delay_rate|     arr_delay_rate|
+-----------------+-----------------+-------------------+-------------------+
|16.32227964287282|16.27152305908301|0.20401663333628903|0.20552087646697423|
+-----------------+-----------------+-------------------+-------------------+



### Airline Performance Comparison

This section compares airline performance by measuring the total number of flights, average arrival delay, and delay rate for each carrier.  
The goal is to identify which airlines experience the highest delay rates and how performance differs across carriers.

In [22]:
spark.sql("""
SELECT
    Reporting_Airline,
    COUNT(*) AS total_flights,
    AVG(ArrDelayMinutes) AS avg_arr_delay,
    SUM(is_arr_delayed) / COUNT(*) AS delay_rate
FROM airline_data
GROUP BY Reporting_Airline
ORDER BY delay_rate DESC
LIMIT 10
""").show()

+-----------------+-------------+------------------+-------------------+
|Reporting_Airline|total_flights|     avg_arr_delay|         delay_rate|
+-----------------+-------------+------------------+-------------------+
|               F9|       202866|24.582507665158282| 0.2857797758126054|
|               AA|       971564|23.494062151335374|0.25626206817049624|
|               B6|       235992|21.306823960134242|0.25226278856910406|
|               NK|       248901| 18.22827951675566|0.23751210320569222|
|               OH|       223343|20.111577260088744|0.21747715397393247|
|               AS|       241900|12.692224059528732|0.21628772219925588|
|               G4|       115686| 19.90275400653493| 0.2144771190982487|
|               MQ|       278679|14.579993469188565|0.20417397794595216|
|               WN|      1409366|12.383495841392513|0.20195321868130778|
|               UA|       749879|15.589153716799643| 0.1952301637997597|
+-----------------+-------------+------------------

### Airport Congestion Analysis

This section examines departure airports to identify which airports experience the most frequent delays.  
By comparing traffic volume, average departure delay, and delay rate, we can better understand congestion patterns at different airports.

In [23]:
spark.sql("""
SELECT
    Origin,
    COUNT(*) AS total_flights,
    AVG(DepDelayMinutes) AS avg_dep_delay,
    SUM(is_dep_delayed) / COUNT(*) AS delay_rate
FROM airline_data
GROUP BY Origin
ORDER BY delay_rate DESC
LIMIT 10
""").show()

+------+-------------+------------------+-------------------+
|Origin|total_flights|     avg_dep_delay|         delay_rate|
+------+-------------+------------------+-------------------+
|   VRB|           41|48.292682926829265| 0.4878048780487805|
|   HTS|          422|  44.6303317535545| 0.4312796208530806|
|   EWN|           58| 68.08620689655173|0.41379310344827586|
|   HGR|          288|          36.34375| 0.3715277777777778|
|   ALO|           51|26.627450980392158|0.35294117647058826|
|   CKB|          203|40.206896551724135|0.33004926108374383|
|   OTH|          340| 37.94705882352941|0.32941176470588235|
|   USA|          692|33.664739884393065|0.32514450867052025|
|   EAU|           50|             44.74|               0.32|
|   GUF|           41| 62.63414634146341| 0.3170731707317073|
+------+-------------+------------------+-------------------+



### Delay Patterns by Departure Hour

This section analyzes how departure delays vary by scheduled departure hour.  
It helps reveal whether flights departing during certain times of the day are more likely to experience delays.

In [24]:
spark.sql("""
SELECT
    dep_hour,
    AVG(DepDelayMinutes) AS avg_delay
FROM airline_data
GROUP BY dep_hour
ORDER BY dep_hour
""").show()

+--------+------------------+
|dep_hour|         avg_delay|
+--------+------------------+
|       0|18.707025780413474|
|       1| 19.59441423628275|
|       2|15.618543046357615|
|       3|16.248785228377066|
|       4| 20.06553911205074|
|       5| 9.460136338200133|
|       6| 8.480940773193868|
|       7| 9.187175822850286|
|       8| 9.814214408948835|
|       9|11.043015697817022|
|      10|12.444978433295125|
|      11| 13.64457571319029|
|      12|15.030844155844155|
|      13|16.367149145284817|
|      14| 17.99460018640726|
|      15|19.146688007328176|
|      16|20.533590560589467|
|      17|21.519169581560917|
|      18|23.144932573661688|
|      19|24.151616690632295|
+--------+------------------+
only showing top 20 rows


### Delay Patterns by Day of the Week

This section evaluates how delay behavior changes across different days of the week.  
It helps determine whether certain weekdays or weekends are associated with higher average delays.

In [25]:
spark.sql("""
SELECT
    DayOfWeek,
    AVG(ArrDelayMinutes) AS avg_delay
FROM airline_data
GROUP BY DayOfWeek
ORDER BY DayOfWeek
""").show()

+---------+------------------+
|DayOfWeek|         avg_delay|
+---------+------------------+
|        1|16.602810172799035|
|        2|14.454087080562507|
|        3|14.245572500401918|
|        4|16.345005835435817|
|        5|18.073351307812146|
|        6|15.548388567007422|
|        7|18.354684578836437|
+---------+------------------+



### Seasonal Delay Patterns

This section studies how flight delays vary across seasons.  
Seasonal analysis helps identify whether winter, spring, summer, or fall is associated with more severe delays.

In [26]:
spark.sql("""
SELECT
    season,
    AVG(ArrDelayMinutes) AS avg_delay
FROM airline_data
GROUP BY season
ORDER BY avg_delay DESC
""").show()

+------+------------------+
|season|         avg_delay|
+------+------------------+
|Summer|20.902780029444497|
|Winter|17.289430736375312|
|Spring|16.991898295584825|
|  Fall| 9.813366108098808|
+------+------------------+



### Delay Cause Analysis

This section investigates the major causes of delays, including carrier delay, weather delay, NAS delay, security delay, and late aircraft delay.  
The goal is to understand which factors contribute most to flight disruptions in the dataset.

In [27]:
spark.sql("""
SELECT
    AVG(CarrierDelay) AS carrier_delay,
    AVG(WeatherDelay) AS weather_delay,
    AVG(NASDelay) AS nas_delay,
    AVG(SecurityDelay) AS security_delay,
    AVG(LateAircraftDelay) AS late_aircraft_delay
FROM airline_data
""").show()

+-----------------+------------------+-----------------+--------------------+-------------------+
|    carrier_delay|     weather_delay|        nas_delay|      security_delay|late_aircraft_delay|
+-----------------+------------------+-----------------+--------------------+-------------------+
|5.254456843976241|0.8991926099823548|2.904462769695556|0.026393125132539545|  6.189231174506264|
+-----------------+------------------+-----------------+--------------------+-------------------+



### Delay Causes by Airline

This section compares delay causes across airlines.  
It helps show whether some airlines are more affected by carrier-related delays, weather-related delays, or late aircraft issues than others.

In [ ]:
spark.sql("""
SELECT
    Reporting_Airline,
    AVG(CarrierDelay) AS carrier_delay,
    AVG(WeatherDelay) AS weather_delay,
    AVG(LateAircraftDelay) AS late_aircraft_delay
FROM airline_data
GROUP BY Reporting_Airline
ORDER BY carrier_delay DESC
LIMIT 10
""").show()